## Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.dbutils import DBUtils
from datetime import datetime
from io import BytesIO
import pandas as pd
import boto3
import json
import os

In [0]:
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        if aws_secret_key is not None:
            print("AWS secret key retrieved successfully.")
        else:
            print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']

# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  

    env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env_prefix

## Get the needed Perimeter and Parameters

In [0]:
env = get_env_prefix()

In [0]:
try:    
    report_args = dbutils.widgets.get("report_args")
    report_args = report_args.replace("\'", "\"")
except Exception as e:
    raise ValueError("Missing required input: report_args")

# Read JSON
json_data = json.loads(report_args)
 
# Access the values and store them into variables
if "output_file_prefix" not in json_data: raise KeyError("'output_file_prefix' key is missing in the report_args JSON data.")
if "report_output" not in json_data: raise KeyError("'report_output' key is missing in the report_args JSON data.")

output_file_prefix = "".join(json_data['output_file_prefix'])
report_output = json_data['report_output']

In [0]:
run_date = dbutils.widgets.get("run_date")

run_date = spark.sql(f"""                    
                        select 
                            case 
                                when try_cast('{run_date}' as date) is not null then try_cast('{run_date}' as date)
                                else PreviousBusinessDay
                            end as BusinessDate
                        from `{env}-cdh`.reference.ref_calendar
                        where CalendarDate = coalesce(try_cast('{run_date}' as date),cast(getdate() as date))
                    """).first()[0]

In [0]:
# Read the table into a Pandas DataFrame
perimeter_path = f"`{env}-reportinputfiles`.internal.legalpoolquarterperimeter"
perimeter_pd = spark.read.table(perimeter_path).toPandas()
perimeter_pd['VatNumber'] = perimeter_pd['VatNumber'].astype(str).str.zfill(9)

# Convert the Pandas DataFrame to a Spark DataFrame
perimeter = spark.createDataFrame(perimeter_pd)
perimeter.createOrReplaceTempView("perimeter")

## Read Tables

In [0]:
dexposure = spark.sql(f"""                    
                    select distinct BusinessDate, ExposureKey, ProductKey, ExposureSourceSystemKeyQCCepal, IsActiveFlag, AssetSegment
                    from `{env}-cdh`.derivative.get_all_exposures_by_business_date('{run_date}')
                    """)

dexposure.createOrReplaceTempView("dexposure")

In [0]:
InitialExposure = spark.sql(f"""                    
                                select distinct BusinessDate, ExposureKey, PartyTaxRegistrationNumber, AgreementKey, PartyRelationshipType, CDI, Sourcesystem, PortfolioName, ProductKey
                                from `{env}-cdh`.primary.exposure
                            """)

InitialExposure.createOrReplaceTempView("InitialExposure")         

In [0]:
All_exposure = spark.sql(f"""                    
                            select distinct ex.BusinessDate, ex.ExposureKey, PartyTaxRegistrationNumber, AgreementKey, PartyRelationshipType, CDI, Sourcesystem, PortfolioName, dex.ProductKey, dex.ExposureSourceSystemKeyQCCepal, dex.IsActiveFlag, dex.AssetSegment
                            from InitialExposure ex
                            inner join dexposure dex
                            on dex.ExposureKey = ex.ExposureKey
                            and ex.BusinessDate = dex.BusinessDate
                            where PortfolioName = 'Mercury'
                        """)

All_exposure.createOrReplaceTempView("All_exposure")

In [0]:
All_exposure_subproduct = spark.sql(f"""                    
                                        SELECT ExposureKey, ProductKey
                                        FROM (
                                            SELECT ExposureKey, ProductKey,
                                                    ROW_NUMBER() OVER (PARTITION BY exposurekey ORDER BY businessdate DESC) AS rn
                                            FROM InitialExposure
                                            WHERE sourcesystem = 'qc'
                                        ) t
                                        WHERE rn = 1
                                    """)

All_exposure_subproduct.createOrReplaceTempView("All_exposure_subproduct")

In [0]:
exposure_perim = spark.sql(f"""                    
                    select distinct ex.BusinessDate, ex.ExposureKey, VatNumber
                    from All_exposure ex
                    inner join perimeter
                    on ex.PartyTaxRegistrationNumber = perimeter.VatNumber
                    """)

exposure_perim.createOrReplaceTempView("exposure_perim")

In [0]:
exposure = spark.sql(f"""                    
                    select distinct ex.BusinessDate, ex.ExposureKey, VatNumber, PartyTaxRegistrationNumber, ExposureSourceSystemKeyQCCepal, AgreementKey, IsActiveFlag, AssetSegment, PartyRelationshipType, CDI, Sourcesystem, PortfolioName, ex.ProductKey, sub.ProductKey as SubProductKey
                    from All_exposure ex
                    inner join exposure_perim
                    on exposure_perim.ExposureKey = ex.ExposureKey
                    left join All_exposure_subproduct sub
                    on exposure_perim.ExposureKey = sub.ExposureKey
                    """)

exposure.createOrReplaceTempView("exposure")

In [0]:
MultiplePortfolios = spark.sql(f"""                    
                                    select distinct ex1.PartyTaxRegistrationNumber
                                    from InitialExposure ex1
                                    inner join perimeter
                                    on ex1.PartyTaxRegistrationNumber = perimeter.VatNumber
                                    inner join (
                                        select distinct ex2.PartyTaxRegistrationNumber
                                        from InitialExposure ex2
                                        where ex2.portfolioname <> 'Mercury'
                                    ) ex2
                                    on ex1.PartyTaxRegistrationNumber = ex2.PartyTaxRegistrationNumber
                                    where ex1.portfolioname = 'Mercury'
                                """)

MultiplePortfolios.createOrReplaceTempView("MultiplePortfolios")

In [0]:
dparty = spark.sql(f"""
                     select distinct exposure.VatNumber, dparty.TaxRegistrationNumber, PartyKeyQCCepal, PartyKeyProfits, PartyKeyThesis, PartyKeyArotron, dparty.CDI, DeceasedFlag, PartyType, Segment, LegalNameGR, LastNameGR, FirstNameGR, FirstFatherNameGR, FirstMotherNameGR, BirthDate, dparty.PortfolioName
                     from `{env}-cdh`.derivative.dparty
                     inner join exposure
                     on dparty.TaxRegistrationNumber = exposure.PartyTaxRegistrationNumber
                     where dparty.BusinessDate = '{run_date}'
                     and dparty.PortfolioName = 'Mercury'
                  """)

dparty.createOrReplaceTempView("dparty")

In [0]:
Agreement = spark.sql(f"""
                        select distinct dagr.AgreementKey, IssueDate
                        from `{env}-cdh`.derivative.DAgreement dagr
                        inner join exposure ex
                        on ex.AgreementKey = dagr.AgreementKey
                        """)

Agreement.createOrReplaceTempView("Agreement")

In [0]:
MasterXSlave = spark.sql(f"""
                        select distinct EntityKey, RelatedEntityKey
                        from `{env}-cdh`.derivativeactive.dentityrelationships dentrel
                        inner join exposure ex
                        on ex.exposurekey = dentrel.RelatedEntityKey
                        where dentrel.BusinessDate = ex.BusinessDate
                        and EntityRelationshipType = 'Master - Slave Rltnp'
                        """)

MasterXSlave.createOrReplaceTempView("MasterXSlave")

In [0]:
DUnderlaw = spark.sql(f"""
                        select distinct TaxRegistrationNumber, CDI, InLawFlag
                        from `{env}-cdh`.derivative.DUnderlaw
                        where UnderLawType = 'N3869'
                        and BusinessDate = '{run_date}'
                        """)

DUnderlaw.createOrReplaceTempView("DUnderlaw")

In [0]:
Assignment = spark.sql(f"""
                            select distinct ExposureKey, AssignmentProcess, ExternalCompanyFullName, Channel, BusinessDate
                            from `{env}-cdh`.primary.Assignment
                            where PortfolioName = 'Mercury'
                            and BusinessDate = '{run_date}'
                        """)

Assignment.createOrReplaceTempView("Assignment")

In [0]:
Collateral = spark.sql(f"""                    
                    select distinct col.Agreementkey, RealEstateKey, col.PortfolioName
                    from `{env}-cdh`.derivative.dcollateral col
                    inner join exposure ex
                    on col.Agreementkey = ex.Agreementkey
                    where col.BusinessDate = '{run_date}'
                    and CollateralType = 'RE'
                    and col.PortfolioName = 'Mercury'
                    """)

Collateral.createOrReplaceTempView("Collateral")

In [0]:
RealEstateInfo = spark.sql(f"""                    
                    select distinct TaxRegistrationNumber
                    from `{env}-cdh`.primary.RealEstateInfo
                    inner join dparty
                    on ownerpartykey = PartyKeyThesis or ownerpartykey = PartyKeyArotron
                    inner join Collateral
                    on realestateinfo.RealEstateKey = collateral.RealEstateKey
                    and Collateral.PortfolioName = 'Mercury'
                    where BusinessDate = '{run_date}'
                    """)

RealEstateInfo.createOrReplaceTempView("RealEstateInfo")

In [0]:
Product = spark.sql(f"""
                     select distinct ProductKey, ProductDescription, ProductCategory, SourceSystem
                     from `{env}-cdh`.primary.product
                     where product.BusinessDate = '{run_date}'
                    """)

Product.createOrReplaceTempView("Product")

In [0]:
Product_profits = spark.sql(f"""
                                select distinct Exposurekey, ProductDescription
                                from product p
                                inner join exposure ex
                                on p.Productkey = ex.Productkey
                                where p.SourceSystem = 'profits'
                            """)

Product_profits.createOrReplaceTempView("Product_profits")

In [0]:
SubProduct = spark.sql(f"""
                     select distinct Exposurekey, ProductDescription AS SubProductDesc, ProductCategory
                     from Product p
                     inner join exposure ex
                     on p.Productkey = ex.SubProductKey
                     where p.SourceSystem = 'qc'
                     """)

SubProduct.createOrReplaceTempView("SubProduct")

In [0]:
ContactDetails = spark.sql(f"""
                        select distinct Street, StreetNumber, PostalCode, City, Sourcesystem, AddContactFlag, Portfolioname, PartyKey, CommunicationFlag, IsActiveFlag
                        from `{env}-cdh`.primary.ContactDetails
                        where BusinessDate = '{run_date}'
                        """)

ContactDetails.createOrReplaceTempView("ContactDetails")

In [0]:
ContactDetails_qc = spark.sql(f"""
                        select distinct dparty.CDI, dparty.TaxRegistrationNumber, Street, StreetNumber, replace(regexp_replace(ContactDetails.PostalCode, '^0+', ''), 'N/A', '') as PostalCode, City
                        from ContactDetails
                        inner join dparty
                        on ContactDetails.PartyKey = PartyKeyQCCepal
                        where Sourcesystem = 'qc'
                        and AddContactFlag = True
                        and ContactDetails.Portfolioname = 'Mercury'
                        """)

ContactDetails_qc.createOrReplaceTempView("ContactDetails_qc")

In [0]:
ContactDetails_profits = spark.sql(f"""
                        select distinct dparty.CDI, dparty.TaxRegistrationNumber, Street, StreetNumber, regexp_replace(PostalCode, '^0+', '') as PostalCode, City
                        from ContactDetails
                        inner join dparty
                        on ContactDetails.PartyKey = PartyKeyProfits
                        where Sourcesystem = 'profits'
                        and CommunicationFlag = 1
                        and ContactDetails.Portfolioname = 'Mercury'
                        """)

ContactDetails_profits.createOrReplaceTempView("ContactDetails_profits")

In [0]:
Totaladdress = spark.sql(f"""
                        select dparty.TaxRegistrationNumber, Street, StreetNumber, regexp_replace(PostalCode, '^0+', '') as PostalCode, City
                        from ContactDetails
                        inner join dparty
                        on ContactDetails.PartyKey = PartyKeyProfits
                        where Sourcesystem = 'profits'
                        and PostalCode <> '' and PostalCode <> '99999'
                        and ContactDetails.portfolioname = 'Mercury'
                        UNION
                        select dparty.TaxRegistrationNumber, Street, StreetNumber, regexp_replace(PostalCode, '^0+', '') as PostalCode, City
                        from ContactDetails
                        inner join dparty
                        on ContactDetails.PartyKey = PartyKeyQCCepal
                        where Sourcesystem = 'qc'
                        and regexp_like(PostalCode, '^[0-9]+$') and replace(PostalCode,'0','')<>'' and PostalCode <> '' and PostalCode <> '99999'
                        and IsActiveFlag = True
                        and ContactDetails.Portfolioname = 'Mercury'
                        """)

Totaladdress.createOrReplaceTempView("Totaladdress")

In [0]:
Totaladdress_cleaned = spark.sql(f"""
                                    select Totaladdress.*
                                    from Totaladdress
                                    left join ContactDetails_profits
                                    on Totaladdress.TaxRegistrationNumber = ContactDetails_profits.TaxRegistrationNumber
                                    and Totaladdress.PostalCode = ContactDetails_profits.PostalCode
                                    left join ContactDetails_qc
                                    on Totaladdress.TaxRegistrationNumber = ContactDetails_qc.TaxRegistrationNumber
                                    and Totaladdress.PostalCode = ContactDetails_qc.PostalCode
                                    where ContactDetails_profits.TaxRegistrationNumber is null
                                    and ContactDetails_qc.TaxRegistrationNumber is null
                                """)

Totaladdress_cleaned.createOrReplaceTempView("Totaladdress_cleaned")

In [0]:
# Create a row number based on PartyId and EffectiveDate
windowSpec = Window.partitionBy("TaxRegistrationNumber").orderBy("Street")

Totaladdress_withRank = Totaladdress_cleaned.withColumn("row_number", F.row_number().over(windowSpec))

# Pivot the data
pivoted_address = Totaladdress_withRank.groupBy("TaxRegistrationNumber").pivot("row_number").agg(
    F.first("Street").alias("Address"),
    F.first("StreetNumber").alias("StreetNo"),
    F.first("PostalCode").alias("PostCode"),
    F.first("City").alias("City")
)

# Flatten column names
for col_name in pivoted_address.columns:
    if col_name not in ["TaxRegistrationNumber"]:
        new_col_name = col_name.split('_')[1] + col_name.split('_')[0]
        pivoted_address = pivoted_address.withColumnRenamed(col_name, new_col_name)

pivoted_address.createOrReplaceTempView("pivoted_address")

## Select Fields

In [0]:
Legal_Pool_Quarter = spark.sql(f"""                    
                                select distinct 
                                      dparty.TaxRegistrationNumber AS TaxNumber
                                    , dparty.CDI
                                    , coalesce(cast(DeceasedFlag as int), 0) AS DeceasedFlag
                                    , PartyType
                                    , coalesce(cast(InLawFlag as int), 0) AS InLawFlag3869
                                    , Segment AS CustomerSegment
                                    , CASE 
                                         WHEN LegalNameGR IS NULL THEN concat(FirstNameGR, ' ', LastNameGR) 
                                         ELSE LegalNameGR
                                       END AS PartyName
                                    , CASE 
                                        WHEN PartyType = 'Organization' THEN NULL
                                        ELSE LastNameGR
                                      END   AS LastName
                                    , CASE 
                                        WHEN PartyType = 'Organization' THEN NULL
                                        ELSE FirstNameGR
                                      END   AS FirstName
                                    , CASE 
                                        WHEN PartyType = 'Organization' THEN NULL
                                        ELSE FirstMotherNameGR
                                      END   AS MotherName
                                    , CASE 
                                        WHEN PartyType = 'Organization' THEN NULL
                                        ELSE FirstFatherNameGR
                                      END   AS FatherName
                                    , CASE 
                                        WHEN date_format(BirthDate, 'd/M/yyyy') = '1/1/1753' THEN CAST(NULL AS STRING)
                                        ELSE date_format(BirthDate, 'd/M/yyyy')
                                      END   AS BirthDate
                                    , CASE 
                                        WHEN MultiplePortfolios.PartyTaxRegistrationNumber IS NOT NULL THEN 1
                                        ELSE 0
                                      END   AS PartyMixedPortfolioFlag
                                    , ex.AgreementKey  AS AgreementNumber
                                    , ex.ExposureKey   AS AccountNumber
                                    , CAST(REPLACE(ex.ExposureKey,'MRC','') AS STRING) AS HostAccountNumber
                                    , ProductDescription AS AccountType
                                    , ProductCategory AS ProductDesc
                                    , SubProductDesc
                                    , CASE 
                                        WHEN IsActiveFlag = true THEN 1
                                        ELSE 0
                                      END   AS IsActiveFlag
                                    , AssetSegment
                                    , date_format(IssueDate, 'd/M/yyyy') as SignatureDate
                                    , MasterXSlave.Entitykey AS MasterAccountNumber
                                    , CASE 
                                        WHEN col.AgreementKey IS NOT NULL THEN 1
                                        ELSE 0
                                      END   AS CollateralIndication
                                    , CASE 
                                        WHEN RealEstateInfo.TaxRegistrationNumber IS NOT NULL THEN 1
                                        ELSE 0
                                      END   AS PropertyOwnershipIndication
                                    , PartyRelationshipType RelationType
                                    , CASE 
                                        WHEN Channel = 'External LF' and AssignmentProcess in ('LF_Offer','N/A') THEN ExternalCompanyFullName
                                        ELSE  coalesce(Channel, 'Unassigned')
                                      END   AS OfferChannel
                                    , CASE 
                                        WHEN Channel = 'External LF' and AssignmentProcess in ('LF_Offer','N/A') THEN NULL
                                        ELSE ExternalCompanyFullName
                                      END   AS LegalChannel
                                    , ContactDetails_qc.Street AS ContactAddress
                                    , ContactDetails_qc.StreetNumber AS ContactStreetNo
                                    , ContactDetails_qc.PostalCode AS ContactPostCode
                                    , ContactDetails_qc.City AS ContactCity
                                    , ContactDetails_profits.Street AS CommunicationAddress
                                    , ContactDetails_profits.StreetNumber AS CommunicationStreetNo
                                    , ContactDetails_profits.PostalCode AS CommunicationPostCode
                                    , ContactDetails_profits.City AS CommunicationCity
                                    , pivoted_address.*
                                    , CONCAT_WS(', ', COLLECT_SET(dparty.VatNumber)) AS Vats
                                from dparty
                                inner join exposure ex
                                on dparty.CDI = ex.CDI
                                inner join Agreement agr
                                on ex.Agreementkey = agr.Agreementkey
                                left join MasterXSlave
                                on ex.exposurekey = MasterXSlave.RelatedEntitykey
                                left join Assignment a
                                on ex.exposurekey = a.ExposureKey
                                left join Collateral col
                                on ex.Agreementkey = col.Agreementkey
                                left join RealEstateInfo
                                on RealEstateInfo.TaxRegistrationNumber = dparty.VatNumber
                                left join MultiplePortfolios
                                on dparty.VatNumber = MultiplePortfolios.PartyTaxRegistrationNumber
                                left join Product_profits
                                on Product_profits.Exposurekey = ex.Exposurekey
                                left join SubProduct
                                on SubProduct.Exposurekey = ex.Exposurekey
                                left join DUnderlaw u
                                on dparty.TaxRegistrationNumber = u.TaxRegistrationNumber
                                or dparty.CDI = u.CDI
                                left join ContactDetails_qc
                                on dparty.TaxRegistrationNumber = ContactDetails_qc.TaxRegistrationNumber
                                and dparty.CDI = ContactDetails_qc.CDI
                                left join ContactDetails_profits
                                on dparty.TaxRegistrationNumber = ContactDetails_profits.TaxRegistrationNumber
                                and dparty.CDI = ContactDetails_profits.CDI
                                left join pivoted_address
                                on dparty.TaxRegistrationNumber = pivoted_address.TaxRegistrationNumber
                                group by all
                              """)
Legal_Pool_Quarter.drop('TaxRegistrationNumber')

string_cols = [c for c, t in Legal_Pool_Quarter.dtypes if t == 'string']
Legal_Pool_Quarter = Legal_Pool_Quarter.replace('', None, subset=string_cols)

Legal_Pool_Quarter.createOrReplaceTempView("Legal_Pool_Quarter")

## Export to Volume and Table

In [0]:
# notebook: {env}-cdh-delivery
# volume: {env}-cdh-delivery.internalreports.legalpoolquarter
# export path: \\dwhsideapps\Share\in\{env}\InternalReports\LegalPoolQuarter

In [0]:
%pip install openpyxl

In [0]:
run_date_str = datetime.now().strftime("%Y%m%d")

#Define the output directory path
output_dir = f"{report_output}{run_date_str}/"

# Ensure the directory exists
os.makedirs(output_dir, exist_ok=True)

# Define the output file path
output_filename = f"{output_file_prefix}_{run_date.strftime('%Y%m%d')}.xlsx"
output_path = os.path.join(output_dir, output_filename)

In [0]:
# Create an Excel writer object
output_buffer = BytesIO()
with pd.ExcelWriter(output_buffer, engine='openpyxl') as writer:
    # Save the subset to a temporary CSV file
    temp_file = os.path.join(output_dir, f"LegalPoolQuarter.csv")
    Legal_Pool_Quarter.coalesce(1).write.csv(temp_file, header=True, mode="overwrite")
    
    # Read the CSV back into Pandas
    csv_files = [f for f in os.listdir(temp_file) if f.endswith(".csv")]
    if csv_files:
        pandas_df = pd.read_csv(os.path.join(temp_file, csv_files[0]))
        
        # Write to an Excel sheet
        sheet_name = 'LegalPoolQuarter'
        pandas_df.to_excel(writer, sheet_name=sheet_name, index=False)
    for file in os.listdir(temp_file):
        os.remove(os.path.join(temp_file, file))

with open(output_path, "wb") as f:
    f.write(output_buffer.getvalue())